# E2 · `predict_proba[:, 1]` is not always the probability you think

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Find out why a model that flags at-risk customers "finds" 94% of the customers (far too many) with a precision of 14%, despite an excellent AUC: the columns of `predict_proba` are
in **alphabetical** order of the classes, and with text labels column 1 is not the one the colleague thinks. Fix it and write a robust function.

## Dataset

`ds.loans(with_missing=False)` with the target turned into text: `"defaulted"` or `"regular"` (`y_tr`, `y_te` are arrays of strings). Features: `ratio, age, job_tenure, income`, logistic with scaler.

## Educational purpose

Recognize one of the most insidious bugs with non-numeric labels: the AUC computed with `roc_auc_score` on the strings is **right by chance** (the positive class is the last one alphabetically, like column 1),
and hides the error, which shows up only when you use the probability to decide.

## Task

1. Run the colleague's cell: `auc_bug` is good, but `bug_flagged_share` (share of customers with `p >= 0.3`) is very high and `bug_precision` very low.
2. Inspect `model.classes_`: save `classes` (list) and `defaulted_index` (position of `"defaulted"` in the list).
3. Recompute with the right column `p_def`: save `ok_flagged_share`, `ok_precision` and `ok_recall` (with threshold 0.3 and `"defaulted"` as the positive class).
4. Write `class_proba(model, X, cls)` that returns the probability of the requested class by looking up its column in `model.classes_`; try it on a model trained with `1/0` labels (class `1`) and on one with reversed text labels (`"yes"` for defaulted and `"no"`, class `"yes"`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
cols = ["ratio", "age", "job_tenure", "income"]
y_txt = np.where(df["defaulted"] == 1, "defaulted", "regular")
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], y_txt, test_size=0.3, stratify=y_txt, random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)

## Your work

### The colleague's code

In [ ]:
p = model.predict_proba(X_te)[:, 1]                                         # "default probability"
auc_bug = roc_auc_score(y_te, p)
flagged = p >= 0.3                                                          # flag as defaulter whoever has p >= 0.3
pred = np.where(flagged, "defaulted", "regular")
bug_flagged_share = flagged.mean()
bug_precision = precision_score(y_te, pred, pos_label="defaulted")
print(f"AUC {auc_bug:.3f} | customers flagged {bug_flagged_share:.1%} | precision {bug_precision:.3f}")

### Your diagnosis

In [ ]:
classes = None
defaulted_index = None

In [ ]:
p_def = None
ok_flagged_share = None
ok_precision = None
ok_recall = None

In [ ]:
def class_proba(model, X, cls):
    ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`p_def = model.predict_proba(X_te)[:, defaulted_index]`. In the pipeline `classes_` is exposed by the final estimator: `model.classes_` works, or `model[-1].classes_`. `class_proba`: `return model.predict_proba(X)[:, list(model.classes_).index(cls)]`.'):
    check(auc_bug > 0.78, "the colleague's AUC is good: the bug hides", "don't modify the colleague's code")
    check(bug_flagged_share > 0.9 and bug_precision < 0.2, "the colleague flags over 90% of the customers with a precision below 20%", "don't modify the colleague's code")
    check(classes == ["defaulted", "regular"] and defaulted_index == 0, "the classes are in alphabetical order: 'defaulted' is column 0", "model.classes_")
    check(close(p_def, model.predict_proba(X_te)[:, 0]), "the default probability is column 0", "use defaulted_index")
    check(ok_flagged_share < 0.25 and ok_precision > 0.5 and ok_recall > 0.5, "with the right column: about 18% flagged, precision and recall above 0.5", "threshold 0.3 on p_def, positive class 'defaulted'")
    m1 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, (y_tr == "defaulted").astype(int))
    m2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, np.where(y_tr == "defaulted", "yes", "no"))
    check(close(class_proba(m1, X_te, 1), m1.predict_proba(X_te)[:, 1]) and close(class_proba(m2, X_te, "yes"), m2.predict_proba(X_te)[:, 1]) and close(class_proba(m2, X_te, "yes"), 1 - class_proba(m2, X_te, "no")),
          "class_proba works with numeric and text labels, in any order", "look up the column with list(model.classes_).index(cls)")
done()

---
The commented solution is in `solutions/E_find_the_bug/E2_predict_proba_columns_and_classes.ipynb`. Open it after you pass the check, or after you have really tried.